# **XGBoost 시계열 예측 파이프라인 구조 요약**

---

## **1. 전체 파이프라인**

1. **데이터 로드**
   - `train`, `val`, `test` CSV 파일 로드  
   - `일시` 컬럼을 `datetime` 형식으로 변환하여 시계열로 처리

2. **테이블 빌드**
   - 시간 특징(`hour`, `doy`, sin/cos) 생성  
   - `lag_k` 및 `diff_past_k` 생성  
   - 다음 24시간 합계 타깃(`target_next_24h`) 생성  
   - 결측 제거 및 0으로 채움  
   → `build_table()`에서 통합 수행

3. **지역별 스케일링 학습**
   - 입력 특징: `StandardScaler → MinMaxScaler`  
   - 타깃: `log1p` 후 `StandardScaler`  
   - 지역 단위 스케일러 세트(`region_scalers`) 생성 및 저장

4. **스케일 변환 적용**
   - `transform_frame()`을 통해 지역별 스케일러를  
     `train`, `val`, `test` 데이터에 적용

5. **Plant 1차 학습**
   - 발전소 단위로 XGBoost 모델 학습  
   - 에포크 단위 부스팅 + EMA 기반 조기 종료  
   - Best 모델(`best.json`) 저장  
   → `train_epochal()` 사용

6. **Region 파인튜닝(2차 학습)**
   - 지역 단위로 fine-tuning 수행  
   - 성능이 가장 높은 발전소 모델로 warm start  
   - 동일한 부스팅 및 조기 종료 전략 사용

7. **시간대 편향 보정**
   - `compute_hour_bias()`로 검증 구간의 시간(`hour`)별 평균 잔차 계산  
   - `apply_hour_bias()`로 시간대별 보정 적용 (보정량은 `CALIB_CLIP` 비율 제한)

8. **검증 성능 요약**
   - 발전소(`df_plant`), 지역(`df_region`), 지역 보정(`df_region_post`) 단위의 성능 표 생성  
   - R², RMSE, MAE 포함

9. **테스트 평가**
   - 발전소별 모델 및 지역별 모델로 Test 성능 계산  
   - 보정 전·후 성능 비교

10. **가중 평균 계산**
    - 표본 수 기반 가중 평균 계산  
    - 검증 및 테스트 각각 R², RMSE, MAE 출력  
    → `_weighted_avg()` 사용

11. **체크포인트 관리**
    - 모델 학습 중 생성된 파일 관리  
    - `best.json`, `last.json`, `checkpoint.json` 유지  
    - UTF-16 등 비정상 파일은 자동 제외

---

## **2. 단계별 주요 함수**

| 구분 | 함수명 | 기능 요약 |
|------|---------|------------|
| **테이블 빌드** | `add_time_feats` | `hour`, `doy` 기반 sin/cos 생성 |
|  | `add_lag_only` | 그룹별 lag_k 생성 |
|  | `add_past_deltas` | lag_1 - lag_(k+1) 계산 |
|  | `make_next24_target` | 24시간 합계 타깃 생성 |
|  | `build_table` | 위 모든 단계 일괄 처리 |
| **스케일링** | `transform_frame` | 지역별 스케일러 적용 |
|  | `inverse_target` | 예측값을 MWh 단위로 복원 |
| **모델 학습** | `as_dmatrix` | XGBoost용 DMatrix 변환 |
|  | `train_epochal` | 에포크 기반 부스팅 및 조기 종료 |
|  | `metrics_mwh` | R², RMSE, MAE 계산 |
| **보정 및 평가** | `compute_hour_bias` | 시간대별 잔차 계산 |
|  | `apply_hour_bias` | 보정량 적용 및 음수 방지 |
| **저장 관리** | `save_booster` | 모델 저장 |
|  | `load_booster` | 모델 로드 |
|  | `_file_ok`, `_seems_utf16` | 파일 검증 |
|  | `_clear_dir_keep3` | 불필요한 체크포인트 삭제 |
| **평가 요약** | `_weighted_avg` | 가중 평균 계산 |

---

## **3. 산출물 요약**

| 항목 | 내용 |
|------|------|
| **발전소 성능표** | `df_plant` (발전소 단위 R², RMSE, MAE) |
| **지역 성능표** | `df_region` (지역 단위 R², RMSE, MAE) |
| **지역 보정 성능표** | `df_region_post` (보정 후 R², RMSE, MAE) |
| **모델 체크포인트** | 각 발전소·지역별 `best.json`, `last.json`, `checkpoint.json` |
| **콘솔 출력** | 검증·테스트의 R², RMSE, MAE 가중 평균 결과 |

---

## **4. 주요 파라미터**

| 변수명 | 설명 |
|--------|------|
| `FEATURE_MODE` | 피처 개수 설정 (22 또는 48) |
| `USE_PAST_DELTAS` | diff_past_k 피처 포함 여부 |
| `CALIB_CLIP` | 시간대 보정량 상한 비율 |
| `USE_HOURLY_CALIB` | 시간대별 보정 활성화 여부 |
| `EPOCHS_PLANT / REGION` | 학습 반복 횟수 |
| `CKPT_DIR / SAVE_DIR` | 모델 저장 및 출력 폴더 |


In [ ]:
# -*- coding: utf-8 -*-
"""
XGBoost (1주/2주 선택 + 22/48 특징 선택) → 다음 24시간 '합계(MWh)' 예측
- 입력 특징:
  * 22 모드: 기상(10) + 시간(4) + lag(4) + diff(4) = 22
  * 48 모드: 기상(10) + 시간(4) + 대표 lag 17 + 해당 diff 17 = 48
- 타깃: target_next_24h = [t+1 ... t+24] 합계
- 1차: Plant 학습 / 2차: Region 파인튜닝 (warm start)
- Validation-hour 편향 보정만 적용 (앙상블 없음)
- 최종 출력: Valid, Test 각각 Plant / Region / Region(보정) 가중 평균 R²·RMSE·MAE
"""

import os, glob, time, warnings
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")

# =========================
# 경로/기본 설정
# =========================
TRAIN_CSV = r"C:\ESG_Project1\file\merge_data\train.csv"
VAL_CSV   = r"C:\ESG_Project1\file\merge_data\val.csv"
TEST_CSV  = r"C:\ESG_Project1\file\merge_data\test.csv"

SAVE_BASE = r"C:\ESG_Project1\xgboost\output"
os.makedirs(SAVE_BASE, exist_ok=True)

TIME_COL, GROUP_COL, REGION_COL = "일시", "발전구분", "지역"
TARGET_HOURLY = "합산발전량(MWh)"
TARGET_NEXT24 = "target_next_24h"

WEATHER_COLS = [
    "기온(°C)", "강수량(mm)", "풍속(m/s)", "습도(%)", "증기압(hPa)",
    "일조(hr)", "일사(MJ/m2)", "적설(cm)", "전운량(10분위)", "중하층운량(10분위)"
]
TIME_FEATS = ["hour_sin", "hour_cos", "doy_sin", "doy_cos"]

# =========================
# 컨텍스트 / 피처 스위치
# =========================
CONTEXT_2W = False          # False: 1주 컨텍스트, True: 2주
FEATURE_MODE = 22           # {22, 48}
DENSE_2W = False            # 2주 컨텍스트일 때 1~336 dense lag 사용
USE_DELTAS = True           # diff 사용 여부 (48 모드는 자동 True)

def _lags_for_48():
    return [1, 2, 3, 6, 12, 18, 24, 36, 48, 72, 96, 120, 144, 168, 240, 288, 336]

def _resolve_context_lags(context_2w: bool, dense_2w: bool):
    if not context_2w:
        return [1, 3, 6, 24]
    if dense_2w:
        return list(range(1, 337))
    return _lags_for_48()

LAGS_CONTEXT = _resolve_context_lags(CONTEXT_2W, DENSE_2W)

if FEATURE_MODE == 22:
    LAGS_FEATURE = [1, 3, 6, 24]
    USE_PAST_DELTAS = True if USE_DELTAS else False
elif FEATURE_MODE == 48:
    LAGS_FEATURE = _lags_for_48()
    USE_PAST_DELTAS = True
else:
    raise ValueError("FEATURE_MODE must be 22 or 48")

LAGS_BUILD = sorted(set(LAGS_CONTEXT).union(LAGS_FEATURE))

CFG_TAG = f"{'2w' if CONTEXT_2W else '1w'}_feat{FEATURE_MODE}_{'dense' if (CONTEXT_2W and DENSE_2W) else 'rep'}"
SAVE_DIR  = os.path.join(SAVE_BASE, CFG_TAG)
CKPT_DIR  = os.path.join(SAVE_DIR, "checkpoints")
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

print(f"🔧 CONTEXT_2W={CONTEXT_2W}  FEATURE_MODE={FEATURE_MODE}  DENSE_2W={DENSE_2W}  USE_PAST_DELTAS={USE_PAST_DELTAS}")
print(f"🔧 LAGS_CONTEXT size={len(LAGS_CONTEXT)}  LAGS_FEATURE size={len(LAGS_FEATURE)}  LAGS_BUILD size={len(LAGS_BUILD)}")

# =========================
# 스위치(성능 옵션)
# =========================
RESUME = True
USE_HOURLY_CALIB = True     # 시간대별 평균잔차 보정
CALIB_CLIP = 0.25           # 보정량 안전 클립 비율

# =========================
# 하이퍼파라미터 (XGBoost)  ← CNN-BiLSTM 감각에 맞춰 Region 길이만 10 epoch
# =========================
EPOCHS_PLANT = 50
BOOSTS_PER_EPOCH_PLANT = 400
XGB_PARAMS_PLANT = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "learning_rate": 1e-3,
    "max_depth": 6,
    "min_child_weight": 6,
    "subsample": 0.7,
    "colsample_bytree": 0.7,
    "gamma": 1.0,
    "lambda": 2.0,
    "reg_alpha": 0.5,
    "tree_method": "hist",
}

EPOCHS_REGION = 10             # CNN-BiLSTM 기준에 맞춰 짧게
BOOSTS_PER_EPOCH_REGION = 800
XGB_PARAMS_REGION = {
    **XGB_PARAMS_PLANT,
    "learning_rate": 1e-3,     # 동일 감각 유지
    "max_depth": 8,
    "min_child_weight": 8,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "lambda": 4.0,
    "reg_alpha": 0.8,
}
PATIENCE_EPOCHS = 10

EMA_ALPHA = 0.5
MIN_DELTA = 1e-4

# =========================
# 유틸
# =========================
def add_time_feats(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.sort_values([GROUP_COL, TIME_COL], inplace=True)
    df["hour"] = df[TIME_COL].dt.hour
    df["doy"]  = df[TIME_COL].dt.dayofyear
    df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
    df["doy_sin"]  = np.sin(2*np.pi*df["doy"]/365)
    df["doy_cos"]  = np.cos(2*np.pi*df["doy"]/365)
    return df

def add_lag_only(df: pd.DataFrame, lags) -> pd.DataFrame:
    df = df.copy()
    df.sort_values([GROUP_COL, TIME_COL], inplace=True)
    lags_ext = sorted(set(lags + [k+1 for k in lags]))
    for k in lags_ext:
        df[f"lag_{k}"] = df.groupby(GROUP_COL)[TARGET_HOURLY].shift(k)
    return df

def add_past_deltas(df: pd.DataFrame, lags, use_deltas: bool) -> pd.DataFrame:
    df = df.copy()
    if not use_deltas:
        return df
    df.sort_values([GROUP_COL, TIME_COL], inplace=True)
    for k in lags:
        lk = f"lag_{k+1}"
        if "lag_1" in df.columns and lk in df.columns:
            df[f"diff_past_{k}"] = df["lag_1"] - df[lk]
    return df

def make_next24_target(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    g = df.groupby(GROUP_COL)[TARGET_HOURLY]
    df[TARGET_NEXT24] = (
        g.shift(-1).rolling(window=24, min_periods=24).sum().reset_index(level=0, drop=True)
    )
    return df

def build_table(df: pd.DataFrame) -> pd.DataFrame:
    df = add_time_feats(df)
    df = add_lag_only(df, lags=LAGS_BUILD)
    df = add_past_deltas(df, lags=LAGS_BUILD, use_deltas=USE_PAST_DELTAS)
    df = make_next24_target(df)
    df.dropna(subset=[TARGET_NEXT24], inplace=True)
    df.fillna(0.0, inplace=True)
    return df

def as_dmatrix(X, y=None):
    return xgb.DMatrix(X, label=y) if y is not None else xgb.DMatrix(X)

# =========================
# 데이터 로드/구성
# =========================
train_raw = pd.read_csv(TRAIN_CSV, parse_dates=[TIME_COL])
val_raw   = pd.read_csv(VAL_CSV,   parse_dates=[TIME_COL])
test_raw  = pd.read_csv(TEST_CSV,  parse_dates=[TIME_COL])

train_raw = build_table(train_raw)
val_raw   = build_table(val_raw)
test_raw  = build_table(test_raw)

lag_cols_all  = [f"lag_{k}" for k in LAGS_FEATURE]
diff_cols_all = [f"diff_past_{k}" for k in LAGS_FEATURE] if USE_PAST_DELTAS else []
all_candidates = WEATHER_COLS + TIME_FEATS + lag_cols_all + diff_cols_all
feature_cols = [c for c in all_candidates if c in train_raw.columns]

print(f"✅ feature_cols ({len(feature_cols)}): {feature_cols[:16]}{' ...' if len(feature_cols)>16 else ''}")

# =========================
# 지역별 스케일러
# =========================
region_scalers = {}
for region, grp in train_raw.groupby(REGION_COL):
    X = grp[feature_cols].to_numpy(np.float32)
    y = np.log1p(grp[[TARGET_NEXT24]].clip(lower=0.0) + 1e-8)
    std = StandardScaler().fit(X)
    mm  = MinMaxScaler().fit(std.transform(X))
    tsc = StandardScaler().fit(y)
    region_scalers[region] = (std, mm, tsc)

def transform_frame(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for region, grp in df.groupby(REGION_COL):
        if region not in region_scalers:
            continue
        std, mm, tsc = region_scalers[region]
        X = grp[feature_cols].to_numpy(np.float32)
        y = np.log1p(grp[[TARGET_NEXT24]].clip(lower=0.0) + 1e-8)
        df.loc[grp.index, feature_cols] = mm.transform(std.transform(X))
        df.loc[grp.index, TARGET_NEXT24] = tsc.transform(y)
    return df

train = transform_frame(train_raw)
val   = transform_frame(val_raw)
test  = transform_frame(test_raw)

# =========================
# 타깃 역변환 및 메트릭
# =========================
def inverse_target(region: str, y_scaled):
    _, _, tsc = region_scalers[region]
    y_log = tsc.inverse_transform(np.asarray(y_scaled).reshape(-1,1)).ravel()
    y = np.expm1(y_log) - 1e-8
    return np.clip(y, 0, None)

def metrics_mwh(region: str, y_true_scaled, y_pred_scaled):
    y_true = inverse_target(region, y_true_scaled)
    y_pred = inverse_target(region, y_pred_scaled)
    r2  = max(r2_score(y_true, y_pred), 0.0)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    return r2, rmse, mae, y_true, y_pred

# =========================
# 안전 저장/재개 유틸
# =========================
def _file_ok(path: str) -> bool:
    try:
        return os.path.exists(path) and os.path.getsize(path) > 0
    except:
        return False

def _seems_utf16(path: str) -> bool:
    try:
        with open(path, "rb") as f:
            head = f.read(8)
        return b"\x00" in head
    except:
        return False

def save_booster(booster: xgb.Booster, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True
    )
    booster.save_model(path)

def load_booster(path: str):
    if not _file_ok(path) or _seems_utf16(path):
        return None
    try:
        booster = xgb.Booster()
        booster.load_model(path)
        return booster
    except:
        return None

def _clear_dir_keep3(dirpath: str):
    keep = {"best.json", "last.json", "checkpoint.json"}
    if not os.path.isdir(dirpath):
        return
    for p in glob.glob(os.path.join(dirpath, "*")):
        base = os.path.basename(p)
        if base not in keep and os.path.isfile(p):
            try:
                os.remove(p)
            except:
                pass

# =========================
# 에포크 학습 루틴 (XGBoost)
# =========================
def train_epochal(
    name,
    dtrain,
    dvalid,
    region,
    y_valid_scaled,
    work_dir,
    epochs,
    boosts_per_epoch,
    params,
    patience_epochs,
    init_model_path=None,
    ema_alpha=EMA_ALPHA,
    min_delta=MIN_DELTA,
):
    os.makedirs(work_dir, exist_ok=True)
    path_best = os.path.join(work_dir, "best.json")
    path_last = os.path.join(work_dir, "last.json")
    path_ckpt = os.path.join(work_dir, "checkpoint.json")

    ema = None
    best_ema = -float("inf")
    best_rounds = 0
    early_counter = 0
    booster = None

    for cand in [path_ckpt, path_last, init_model_path]:
        if cand and _file_ok(cand) and not _seems_utf16(cand):
            booster = load_booster(cand)
            if booster is not None:
                print(f"🔄 Resume from: {cand}")
                break

    wall_start = time.time()
    base_rounds = booster.num_boosted_rounds() if booster is not None else 0
    total_target_rounds = epochs * boosts_per_epoch

    for ep in range(1, epochs+1):
        booster = xgb.train(
            params=params,
            dtrain=dtrain,
            num_boost_round=boosts_per_epoch,
            xgb_model=booster,
            evals=[(dvalid, "valid")],
            verbose_eval=False
        )
        now_rounds = booster.num_boosted_rounds()

        try:
            y_pred_scaled = booster.predict(dvalid, iteration_range=(0, now_rounds))
        except TypeError:
            y_pred_scaled = booster.predict(dvalid)

        r2, rmse, mae, _, _ = metrics_mwh(region, y_valid_scaled, y_pred_scaled)
        ema = r2 if ema is None else (ema_alpha * r2 + (1 - ema_alpha) * ema)

        improved = (ema - best_ema) > min_delta
        best_mark = ""
        if improved:
            best_ema = ema
            best_rounds = now_rounds
            save_booster(booster, path_best)
            best_mark = "★Best★"
            early_counter = 0
        else:
            early_counter += 1

        elapsed = time.time() - wall_start
        done_rounds = max(1, now_rounds - base_rounds)
        avg_per_round = elapsed / done_rounds
        remain_rounds = max(0, total_target_rounds - now_rounds)
        eta = avg_per_round * remain_rounds
        eta_str = f"{eta/60:.2f}m" if eta >= 60 else f"{eta:.1f}s"

        print(f"Epoch {ep}/{epochs} | Val R2={r2:.4f} | EMA R2={ema:.4f} {best_mark} | RMSE={rmse:.4f} | MAE={mae:.4f} | ETA={eta_str}")

        save_booster(booster, path_last)
        save_booster(booster, path_ckpt)
        _clear_dir_keep3(work_dir)

        if early_counter >= patience_epochs:
            print(f"⏹ EarlyStopping Triggered (best_rounds={best_rounds})")
            break

    final_path = path_best if _file_ok(path_best) else path_last
    final = load_booster(final_path)
    if final is not None:
        try:
            y_pred_scaled = final.predict(dvalid, iteration_range=(0, final.num_boosted_rounds()))
        except TypeError:
            y_pred_scaled = final.predict(dvalid)
        r2, rmse, mae, _, _ = metrics_mwh(region, y_valid_scaled, y_pred_scaled)
        print(f"✅ {name} 평가 완료 | R2={r2:.4f} | RMSE={rmse:.4f} | MAE={mae:.4f}")
    else:
        print(f"⚠️ 최종 모델 로드 실패: {final_path}")
        y_pred_scaled = booster.predict(dvalid)
        r2, rmse, mae, _, _ = metrics_mwh(region, y_valid_scaled, y_pred_scaled)
        save_booster(booster, path_best)
        final_path = path_best
        print(f"✅ {name} 평가 완료(대체) | R2={r2:.4f} | RMSE={rmse:.4f} | MAE={mae:.4f}")

    return final_path, (r2, rmse, mae)

# =========================
# 1) Plant 1차 학습
# =========================
plant_models_1st = {}
plant_val_scores = {}

for plant, df_tr in train.groupby(GROUP_COL):
    df_va = val[val[GROUP_COL] == plant]
    if len(df_tr) == 0 or len(df_va) == 0:
        print(f"⚠ {plant}: 데이터 없음. 스킵")
        continue

    X_tr = df_tr[feature_cols].to_numpy(np.float32)
    y_tr = df_tr[TARGET_NEXT24].to_numpy(np.float32)
    X_va = df_va[feature_cols].to_numpy(np.float32)
    y_va = df_va[TARGET_NEXT24].to_numpy(np.float32)
    region = df_va[REGION_COL].iloc[0] if len(df_va) else df_tr[REGION_COL].iloc[0]

    dtrain = as_dmatrix(X_tr, y_tr)
    dvalid = as_dmatrix(X_va, y_va)
    work_dir = os.path.join(CKPT_DIR, "plant", plant)
    init_model = os.path.join(work_dir, "best.json") if (RESUME and os.path.exists(os.path.join(work_dir, "best.json"))) else None

    print(f"\n🌱 Plant 1차 학습 시작: {plant} | train size: {len(df_tr)}")
    if init_model: print(f"🔄 Resume: {init_model}")

    best_path, metrics = train_epochal(
        name=f"Plant {plant}",
        dtrain=dtrain, dvalid=dvalid,
        region=region, y_valid_scaled=y_va,
        work_dir=work_dir,
        epochs=EPOCHS_PLANT, boosts_per_epoch=BOOSTS_PER_EPOCH_PLANT,
        params=XGB_PARAMS_PLANT,
        patience_epochs=10,
        init_model_path=init_model
    )

    plant_models_1st[plant] = best_path
    plant_val_scores[plant]  = dict(region=region, r2=metrics[0], rmse=metrics[1], mae=metrics[2])

# =========================
# 2) Region 파인튜닝 (warm start)
# =========================
region_models = {}
region_metrics = {}
region_seed_path = {}

for region, df_tr_region in train.groupby(REGION_COL):
    df_va_region = val[val[REGION_COL] == region]
    if len(df_tr_region) == 0 or len(df_va_region) == 0:
        print(f"⚠ Region {region}: 데이터 없음. 스킵"); continue

    X_tr = df_tr_region[feature_cols].to_numpy(np.float32)
    y_tr = df_tr_region[TARGET_NEXT24].to_numpy(np.float32)
    X_va = df_va_region[feature_cols].to_numpy(np.float32)
    y_va = df_va_region[TARGET_NEXT24].to_numpy(np.float32)

    dtrain = as_dmatrix(X_tr, y_tr)
    dvalid = as_dmatrix(X_va, y_va)

    plants_in_region = df_tr_region[GROUP_COL].unique().tolist()
    cand = [(p, plant_val_scores.get(p, {}).get("r2", -1)) for p in plants_in_region]
    cand.sort(key=lambda x: x[1], reverse=True)
    seed_model_path = plant_models_1st.get(cand[0][0], None) if cand else None

    work_dir = os.path.join(CKPT_DIR, "region", region)
    if RESUME and os.path.exists(os.path.join(work_dir, "best.json")):
        init_model = os.path.join(work_dir, "best.json")
        print(f"\n🌿 Region {region} 재개")
    else:
        init_model = seed_model_path
        print(f"\n🌿 Region {region} warm start: {init_model}")

    best_path, metrics = train_epochal(
        name=f"Region {region}",
        dtrain=dtrain, dvalid=dvalid,
        region=region, y_valid_scaled=y_va,
        work_dir=work_dir,
        epochs=EPOCHS_REGION, boosts_per_epoch=BOOSTS_PER_EPOCH_REGION,
        params=XGB_PARAMS_REGION,
        patience_epochs=PATIENCE_EPOCHS,
        init_model_path=init_model
    )
    region_models[region] = best_path
    region_metrics[region] = metrics
    region_seed_path[region] = seed_model_path  # Warm-start 기록

# =========================
# 3) Validation-hour 편향 보정
# =========================
def compute_hour_bias(region, booster, df_val_region):
    if df_val_region.empty: return {}
    Xv = df_val_region[feature_cols].to_numpy(np.float32)
    yv = df_val_region[TARGET_NEXT24].to_numpy(np.float32)
    dv = as_dmatrix(Xv)
    y_pred_scaled = booster.predict(dv)
    _, _, _, y_true, y_pred = metrics_mwh(region, yv, y_pred_scaled)
    hours = df_val_region["hour"].to_numpy()
    df_tmp = pd.DataFrame({"hour": hours, "y_true": y_true, "y_pred": y_pred})
    df_tmp["res"] = df_tmp["y_true"] - df_tmp["y_pred"]
    return df_tmp.groupby("hour")["res"].mean().to_dict()

def apply_hour_bias(region, y_pred, hours, bias_dict):
    if not USE_HOURLY_CALIB or not bias_dict: return y_pred
    y_adj = y_pred.copy()
    for i, h in enumerate(hours):
        b = bias_dict.get(int(h), 0.0)
        cap = max(1e-8, abs(y_adj[i]) * CALIB_CLIP)
        b = max(-cap, min(cap, b))
        y_adj[i] = max(0.0, y_adj[i] + b)
    return y_adj

region_post_metrics = {}
region_hour_bias = {}

for region, model_path in region_models.items():
    booster_region = load_booster(model_path)
    if booster_region is None: continue

    df_va_region = val[val[REGION_COL] == region].copy()
    if df_va_region.empty: continue

    Xv = df_va_region[feature_cols].to_numpy(np.float32)
    yv = df_va_region[TARGET_NEXT24].to_numpy(np.float32)
    dv = as_dmatrix(Xv)
    y_pred_r_scaled = booster_region.predict(dv)
    r2_r, rmse_r, mae_r, y_true_r, y_pred_r = metrics_mwh(region, yv, y_pred_r_scaled)

    bias = compute_hour_bias(region, booster_region, df_va_region) if USE_HOURLY_CALIB else {}
    hours = df_va_region["hour"].to_numpy()
    y_pred_r_cal = apply_hour_bias(region, y_pred_r, hours, bias)
    region_hour_bias[region] = bias

    r2_b  = max(r2_score(y_true_r, y_pred_r_cal), 0.0)
    rmse_b= float(np.sqrt(mean_squared_error(y_true_r, y_pred_r_cal)))
    mae_b = float(mean_absolute_error(y_true_r, y_pred_r_cal))

    region_post_metrics[region] = dict(
        r2_region=r2_r, rmse_region=rmse_r, mae_region=mae_r,
        r2_post=r2_b, rmse_post=rmse_b, mae_post=mae_b
    )

# =========================
# 4) 요약 테이블 구성
# =========================
df_plant = pd.DataFrame([
    {"plant": k, "region": v.get("region"), "val_r2": v.get("r2"), "val_rmse": v.get("rmse"), "val_mae": v.get("mae")}
    for k, v in plant_val_scores.items()
])

df_region = pd.DataFrame([
    {"region": r, "val_r2": m[0], "val_rmse": m[1], "val_mae": m[2]}
    for r, m in region_metrics.items()
])

df_region_post = pd.DataFrame([
    {"region": r, **m} for r, m in region_post_metrics.items()
])

# =========================
# 5) Test 평가 (Plant/Region, 보정)
# =========================
plant_test_rows = []
for plant, df_te in test.groupby(GROUP_COL):
    region = df_te[REGION_COL].iloc[0]
    model_path = plant_models_1st.get(plant)
    if not model_path: continue
    booster = load_booster(model_path)
    if booster is None or len(df_te) == 0: continue

    Xte = df_te[feature_cols].to_numpy(np.float32)
    yte = df_te[TARGET_NEXT24].to_numpy(np.float32)
    dte = as_dmatrix(Xte)
    try:
        y_pred_scaled = booster.predict(dte)
        r2, rmse, mae, _, _ = metrics_mwh(region, yte, y_pred_scaled)
        plant_test_rows.append({
            "plant": plant, "region": region,
            "test_r2": r2, "test_rmse": rmse, "test_mae": mae
        })
    except Exception:
        continue

df_plant_test = pd.DataFrame(plant_test_rows)

region_test_raw_rows  = []
region_test_post_rows = []

for region, df_te_region in test.groupby(REGION_COL):
    model_path = region_models.get(region)
    booster_region = load_booster(model_path) if model_path else None
    if booster_region is None or len(df_te_region) == 0: continue

    Xte = df_te_region[feature_cols].to_numpy(np.float32)
    yte = df_te_region[TARGET_NEXT24].to_numpy(np.float32)
    dte = as_dmatrix(Xte)

    y_pred_r_scaled_t = booster_region.predict(dte)
    r2_r_t, rmse_r_t, mae_r_t, y_true_t, y_pred_r_t = metrics_mwh(region, yte, y_pred_r_scaled_t)
    region_test_raw_rows.append({
        "region": region,
        "test_r2": r2_r_t, "test_rmse": rmse_r_t, "test_mae": mae_r_t
    })

    bias_dict = region_hour_bias.get(region, {})
    hours_t   = df_te_region["hour"].to_numpy()
    y_pred_r_cal_t = apply_hour_bias(region, y_pred_r_t, hours_t, bias_dict)

    r2_post_t  = max(r2_score(y_true_t, y_pred_r_cal_t), 0.0)
    rmse_post_t= float(np.sqrt(mean_squared_error(y_true_t, y_pred_r_cal_t)))
    mae_post_t = float(mean_absolute_error(y_true_t, y_pred_r_cal_t))
    region_test_post_rows.append({
        "region": region,
        "r2_post": r2_post_t, "rmse_post": rmse_post_t, "mae_post": mae_post_t
    })

df_region_test_raw  = pd.DataFrame(region_test_raw_rows)
df_region_test_post = pd.DataFrame(region_test_post_rows)

# =========================
# 6) 최종 결과만 출력(가중 평균)
# =========================
def _weighted_avg(values, weights):
    denom = float(weights.sum()) if float(weights.sum()) > 0 else 1.0
    return float((values * weights).sum() / denom)

# Validation 가중 평균
if len(df_plant) > 0:
    w1 = np.array([len(train[train[GROUP_COL]==p]) for p in df_plant["plant"]], dtype=float)
    weighted_r2_1   = _weighted_avg(df_plant["val_r2"].to_numpy(float),   w1)
    weighted_rmse_1 = _weighted_avg(df_plant["val_rmse"].to_numpy(float), w1)
    weighted_mae_1  = _weighted_avg(df_plant["val_mae"].to_numpy(float),  w1)
else:
    weighted_r2_1 = weighted_rmse_1 = weighted_mae_1 = float('nan')

if len(df_region) > 0:
    w2 = np.array([len(train[train[REGION_COL]==r]) for r in df_region["region"]], dtype=float)
    weighted_r2_2   = _weighted_avg(df_region["val_r2"].to_numpy(float),   w2)
    weighted_rmse_2 = _weighted_avg(df_region["val_rmse"].to_numpy(float), w2)
    weighted_mae_2  = _weighted_avg(df_region["val_mae"].to_numpy(float),  w2)
else:
    weighted_r2_2 = weighted_rmse_2 = weighted_mae_2 = float('nan')

if len(df_region_post) > 0:
    w2p = np.array([len(train[train[REGION_COL]==r]) for r in df_region_post["region"]], dtype=float)
    weighted_r2_post   = _weighted_avg(df_region_post["r2_post"].to_numpy(float),   w2p)
    weighted_rmse_post = _weighted_avg(df_region_post["rmse_post"].to_numpy(float), w2p)
    weighted_mae_post  = _weighted_avg(df_region_post["mae_post"].to_numpy(float),  w2p)
else:
    weighted_r2_post = weighted_rmse_post = weighted_mae_post = float('nan')

print("\n===== [VALID] 최종 결과(가중 평균) =====")
print(f"Plant        R²={weighted_r2_1:.4f}  RMSE={weighted_rmse_1:.4f}  MAE={weighted_mae_1:.4f}")
print(f"Region       R²={weighted_r2_2:.4f}  RMSE={weighted_rmse_2:.4f}  MAE={weighted_mae_2:.4f}")
print(f"Region(보정) R²={weighted_r2_post:.4f}  RMSE={weighted_rmse_post:.4f}  MAE={weighted_mae_post:.4f}")

# Test 가중 평균
if len(df_plant_test) > 0:
    w1_t = np.array([len(test[test[GROUP_COL]==p]) for p in df_plant_test["plant"]], dtype=float)
    weighted_r2_1_t   = _weighted_avg(df_plant_test["test_r2"].to_numpy(float),   w1_t)
    weighted_rmse_1_t = _weighted_avg(df_plant_test["test_rmse"].to_numpy(float), w1_t)
    weighted_mae_1_t  = _weighted_avg(df_plant_test["test_mae"].to_numpy(float),  w1_t)
else:
    weighted_r2_1_t = weighted_rmse_1_t = weighted_mae_1_t = float('nan')

if len(df_region_test_raw) > 0:
    w2_t = np.array([len(test[test[REGION_COL]==r]) for r in df_region_test_raw["region"]], dtype=float)
    weighted_r2_2_t   = _weighted_avg(df_region_test_raw["test_r2"].to_numpy(float),   w2_t)
    weighted_rmse_2_t = _weighted_avg(df_region_test_raw["test_rmse"].to_numpy(float), w2_t)
    weighted_mae_2_t  = _weighted_avg(df_region_test_raw["test_mae"].to_numpy(float),  w2_t)
else:
    weighted_r2_2_t = weighted_rmse_2_t = weighted_mae_2_t = float('nan')

if len(df_region_test_post) > 0:
    w2p_t = np.array([len(test[test[REGION_COL]==r]) for r in df_region_test_post["region"]], dtype=float)
    weighted_r2_2p_t   = _weighted_avg(df_region_test_post["r2_post"].to_numpy(float),   w2p_t)
    weighted_rmse_2p_t = _weighted_avg(df_region_test_post["rmse_post"].to_numpy(float), w2p_t)
    weighted_mae_2p_t  = _weighted_avg(df_region_test_post["mae_post"].to_numpy(float),  w2p_t)
else:
    weighted_r2_2p_t = weighted_rmse_2p_t = weighted_mae_2p_t = float('nan')

print("\n===== [TEST] 최종 결과(가중 평균) =====")
print(f"Plant        R²={weighted_r2_1_t:.4f}  RMSE={weighted_rmse_1_t:.4f}  MAE={weighted_mae_1_t:.4f}")
print(f"Region       R²={weighted_r2_2_t:.4f}  RMSE={weighted_rmse_2_t:.4f}  MAE={weighted_mae_2_t:.4f}")
print(f"Region(보정) R²={weighted_r2_2p_t:.4f}  RMSE={weighted_rmse_2p_t:.4f}  MAE={weighted_mae_2p_t:.4f}")

print("\n🏁 완료")


🔧 CONTEXT_2W=False  FEATURE_MODE=48  DENSE_2W=False  USE_PAST_DELTAS=True
🔧 LAGS_CONTEXT size=4  LAGS_FEATURE size=17  LAGS_BUILD size=17
✅ feature_cols (48): ['기온(°C)', '강수량(mm)', '풍속(m/s)', '습도(%)', '증기압(hPa)', '일조(hr)', '일사(MJ/m2)', '적설(cm)', '전운량(10분위)', '중하층운량(10분위)', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'lag_1', 'lag_2'] ...

🌱 Plant 1차 학습 시작: 남제주소내 | train size: 78864
🔄 Resume: C:\ESG_Project1\xgboost\output\1w_feat48_rep\checkpoints\plant\남제주소내\best.json
🔄 Resume from: C:\ESG_Project1\xgboost\output\1w_feat48_rep\checkpoints\plant\남제주소내\checkpoint.json
Epoch 1/50 | Val R2=0.9665 | EMA R2=0.9665 ★Best★ | RMSE=0.0551 | MAE=0.0403 | ETA=1.22m
Epoch 2/50 | Val R2=0.9665 | EMA R2=0.9665  | RMSE=0.0551 | MAE=0.0403 | ETA=1.15m
Epoch 3/50 | Val R2=0.9665 | EMA R2=0.9665  | RMSE=0.0551 | MAE=0.0403 | ETA=59.8s
Epoch 4/50 | Val R2=0.9666 | EMA R2=0.9665  | RMSE=0.0551 | MAE=0.0403 | ETA=49.9s
Epoch 5/50 | Val R2=0.9666 | EMA R2=0.9666  | RMSE=0.0551 | MAE=0.0403 | ETA=39.8s
Epo